In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import sys
print(sys.version)
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("IrisSparkML")
    .getOrCreate()
)


3.10.4 (tags/v3.10.4:9d38120, Mar 23 2022, 23:13:41) [MSC v.1929 64 bit (AMD64)]


In [3]:
df = spark.read.csv("iris.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)

root
 |-- sepal_length: double (nullable = true)
 |-- sepal_width: double (nullable = true)
 |-- petal_length: double (nullable = true)
 |-- petal_width: double (nullable = true)
 |-- species: string (nullable = true)

+------------+-----------+------------+-----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|species|
+------------+-----------+------------+-----------+-------+
|         5.1|        3.5|         1.4|        0.2| setosa|
|         4.9|        3.0|         1.4|        0.2| setosa|
|         4.7|        3.2|         1.3|        0.2| setosa|
|         4.6|        3.1|         1.5|        0.2| setosa|
|         5.0|        3.6|         1.4|        0.2| setosa|
+------------+-----------+------------+-----------+-------+
only showing top 5 rows



In [5]:
feature_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
label_col = "species"
indexer = StringIndexer(inputCol=label_col, outputCol="label")
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
clf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50)

In [6]:
pipeline = Pipeline(stages=[indexer, assembler, clf])

In [7]:
train, test = df.randomSplit([0.7, 0.3], seed=42)
model = pipeline.fit(train)

In [8]:
pred = model.transform(test)

In [9]:
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy",
)
acc = evaluator.evaluate(pred)
print("Test accuracy:", acc)

Test accuracy: 0.9565217391304348


In [ ]:
model.write().overwrite().save("iris_rf_model")
